# Option B -- PC-Well Feature-Space Recentering: Sweep

For every (model, curve_type, outlier_filter, curve_alignment[, pc_ttp_anchor])
combination that has actually been trained, treats each of the group's chips as if
it were a brand-new, unseen chip in turn (using the LOFO fold model that genuinely
excluded that chip from training -- not the `--train_full` model, which would have
already seen it), predicts with and without `--pc_recenter`, and compares against
that chip's real labels.

Reuses `08_cross_dataset_predict_new_chip.py`'s own functions (`align_new_chip`,
`predict_new_chip`, `reference_pc_embedding`) via import -- nothing here is a
reimplementation, same pattern as the earlier calibration notebook. `predict_new_chip`
handles both plain curve models and spatial (`cosine_recon`/`attn_recon`) models
transparently -- for spatial models it builds a real neighbor stack from the chip's
own `coords`/`well_ids` for the main curves, and uses a mean-PC-curve-repeated stack
for `pc_recenter`'s embeddings (no real PC spatial coordinates needed -- see `08`'s
`_pc_mean_stack` docstring for why that's exact, not approximate).

**Combinations that aren't trained yet are skipped, not errored** -- point
`MODELS_TO_TRY`/`OUTLIER_FILTERS_TO_TRY`/`ALIGNMENTS_TO_TRY` at whatever you want to
compare; missing `.keras` files just print `[SKIP]` and the sweep continues. Re-run
this notebook any time (e.g. once the current job finishes) to pick up newly-trained
combinations -- no need to prune the config lists down to only what exists yet.

In [1]:
import os
import importlib
from pathlib import Path

import numpy as np
import pandas as pd

try:
    notebook_path = globals().get('__vsc_ipynb_file__')
    if notebook_path:
        notebook_dir = os.path.dirname(os.path.dirname(notebook_path))
        os.chdir(notebook_dir)
except Exception as e:
    print(f"Could not change directory: {e}")

print("Current Working Directory:", os.getcwd())
%load_ext autoreload
%autoreload 2
import config

# Import-only -- none of these .py files are modified.
cdt = importlib.import_module("04_cross_dataset_training")
p08 = importlib.import_module("08_cross_dataset_predict_new_chip")
vis07 = importlib.import_module("07_attribution_vis_all")

import joblib

Current Working Directory: /vol/bitbucket/gk225/POC_DDM/gk_code/main

[*] SUCCESS: TensorFlow is utilizing the GPU -> /physical_device:GPU:0



## 1. Configuration -- edit these lists to change what gets compared

In [2]:
GROUP_NAME = "final_4_chip_clean"
EXP_FOLDER = config.FINAL_EXP_FOLDER + "_nc_subtract"
MODE_STR = "lofo"

CURVE_TYPES_TO_TRY = ["ori_curve_wavelet_bior35_norm"]

MODELS_TO_TRY = [
    "cnn_gru_dual",                          "cnn_gru_dual_dann",                     "cnn_gru_dual_coral",
    "cnn_gru_dual_supcon3",                  "cnn_gru_dual_supcon3_dann",             "cnn_gru_dual_supcon3_coral",
    "cnn_gru_dual_attn_recon",               "cnn_gru_dual_attn_recon_dann",          "cnn_gru_dual_attn_recon_coral",
    "cnn_gru_dual_attn_recon_supcon3",       "cnn_gru_dual_attn_recon_supcon3_dann",  "cnn_gru_dual_attn_recon_supcon3_coral",
]

OUTLIER_FILTERS_TO_TRY = ["none", "lofo_ae"]

# (curve_alignment, pc_ttp_anchor) -- anchor is ignored when alignment is acquisition_start
ALIGNMENTS_TO_TRY = [
    ("acquisition_start", "min"),
    ("pc_ttp", "min"),
    ("pc_ttp", "percentile"),
]

folder_names = config.CROSS_DATASET_GROUPS[GROUP_NAME]
exp_paths_all = [Path(EXP_FOLDER, name) for name in folder_names]

def short_name(folder):
    return folder.split('_U_', 1)[1]

print(f"{len(folder_names)} chips x {len(MODELS_TO_TRY)} models x {len(OUTLIER_FILTERS_TO_TRY)} filters "
     f"x {len(ALIGNMENTS_TO_TRY)} alignments x {len(CURVE_TYPES_TO_TRY)} curve_types = up to "
     f"{len(folder_names)*len(MODELS_TO_TRY)*len(OUTLIER_FILTERS_TO_TRY)*len(ALIGNMENTS_TO_TRY)*len(CURVE_TYPES_TO_TRY)} rows "
     f"(most will be [SKIP]ped until trained).")

4 chips x 8 models x 2 filters x 3 alignments x 1 curve_types = up to 192 rows (most will be [SKIP]ped until trained).


## 2. Helpers

`out_dir_for` mirrors `04`/`08`'s own alignment-namespacing exactly (reused logic,
not reimplemented -- just the path-join, since `08`'s functions all take `out_dir`
as a parameter rather than deriving it themselves). `_ALIGN_CACHE` avoids re-aligning
the same chip's curves for every model/filter that shares the same
(curve_type, curve_alignment, anchor) -- alignment doesn't depend on either.

In [3]:
def out_dir_for(curve_type, curve_alignment, pc_ttp_anchor):
    out_dir = Path(EXP_FOLDER) / "cross_dataset_cv" / GROUP_NAME
    if curve_alignment == "pc_ttp":
        out_dir = out_dir / "curve_alignment_pc_ttp" / f"anchor_{pc_ttp_anchor}"
    return out_dir


_ALIGN_CACHE = {}

def aligned_chip(chip_path, curve_type, curve_alignment, pc_ttp_anchor):
    key = (chip_path.name, curve_type, curve_alignment, pc_ttp_anchor)
    if key not in _ALIGN_CACHE:
        out_dir = out_dir_for(curve_type, curve_alignment, pc_ttp_anchor)
        _ALIGN_CACHE[key] = p08.align_new_chip(chip_path, out_dir, curve_type, curve_alignment, pc_ttp_anchor)
    return _ALIGN_CACHE[key]


def ground_truth(chip_name, Y_well_raw, class_names):
    mapping = config.LABEL_MAPPINGS[chip_name]
    y_true = np.array([mapping.get(w, w) for w in Y_well_raw])
    # Ground-truth label strings don't always match the model's own class_names
    # (e.g. 'NC-ALL' in LABEL_MAPPINGS vs 'NC' in class_names) -- map by best-effort
    # prefix match against whatever the model actually predicts.
    if class_names:
        cn = list(class_names)
        y_true = np.array([next((c for c in cn if y == c or y.startswith(c + '-') or c.startswith(y + '-')), y)
                           for y in y_true])
    return y_true


print("Helpers defined.")

Helpers defined.


## 3. Sweep

For each (curve_type, alignment, filter, model), for each chip: find the LOFO fold
model that excluded that chip (`model_interpretation/lofo_{chip}/`) -- a genuine
held-out test, not resubstitution against a `--train_full` model that already saw
the chip. Skips (with a printed reason) whenever: the model file doesn't exist yet,
or the chip has no PC snapshot (`--pc_recenter` needs one). `cosine_recon`/`attn_recon`
models are supported -- `predict_new_chip` auto-detects them from the model key and
builds the spatial neighbor-stack itself (same function `08`'s own CLI uses), using
the chip's own `coords`/`well_ids` for the main curves and the mean-PC-curve trick
for `pc_recenter`'s reference/new-chip embeddings (no real PC spatial coords needed).

In [4]:
rows = []

for curve_type in CURVE_TYPES_TO_TRY:
    for curve_alignment, pc_ttp_anchor in ALIGNMENTS_TO_TRY:
        out_dir = out_dir_for(curve_type, curve_alignment, pc_ttp_anchor)
        results_path = out_dir / config.CROSS_DATASET_RESULT_PATH.format(mode=MODE_STR, curve_type=curve_type)
        if not results_path.exists():
            print(f"[SKIP] no results file for curve_type={curve_type} alignment={curve_alignment}/{pc_ttp_anchor} -- not trained yet.")
            continue
        lofo_results = joblib.load(results_path)

        for filter_key_raw in OUTLIER_FILTERS_TO_TRY:
            filter_key = "None" if filter_key_raw.lower() == "none" else filter_key_raw

            for model_key in MODELS_TO_TRY:
                for chip_path in exp_paths_all:
                    chip_name = chip_path.name
                    fold_label = f"lofo_{chip_name}"
                    model_dir = out_dir / "model_interpretation" / fold_label
                    model_path = model_dir / f"{model_key}_{filter_key}_{curve_type}_model.keras"
                    if not model_path.exists():
                        continue  # not trained yet for this fold -- silent skip, too many combos to log each one

                    class_names = lofo_results.get(fold_label, {}).get("class_names")

                    align_result = aligned_chip(chip_path, curve_type, curve_alignment, pc_ttp_anchor)
                    if align_result is None:
                        continue
                    curves, resampler, Y_well_raw, pc_curves_aligned, coords, well_ids = align_result

                    loaded = vis07.load_saved_models(
                        model_dir, filter_key, len(resampler.t_grid), curve_type=curve_type, model_names=[model_key])
                    if model_key not in loaded:
                        continue
                    model = loaded[model_key]

                    if p08._is_spatial(model_key) and (coords is None or well_ids is None):
                        continue  # no spatial metadata for this chip -- can't build a neighbor stack

                    y_true = ground_truth(chip_name, Y_well_raw, class_names)
                    valid = y_true != "PC"

                    exp_paths_train = [p for p in exp_paths_all if p.name != chip_name]

                    probs_base, _ = p08.predict_new_chip(
                        model, model_key, curves, coords, well_ids, pc_curves_aligned,
                        exp_paths_train, out_dir, curve_type, filter_key, curve_alignment,
                        pc_recenter=False)
                    pred_base = np.array(class_names)[np.argmax(probs_base, axis=1)] if class_names else np.argmax(probs_base, axis=1)
                    acc_base = (pred_base[valid] == y_true[valid]).mean() if valid.any() else float('nan')

                    acc_recenter = float('nan')
                    shift_norm = float('nan')
                    try:
                        probs_r, shift_norm = p08.predict_new_chip(
                            model, model_key, curves, coords, well_ids, pc_curves_aligned,
                            exp_paths_train, out_dir, curve_type, filter_key, curve_alignment,
                            pc_recenter=True, force_rerun=True)
                        pred_r = np.array(class_names)[np.argmax(probs_r, axis=1)] if class_names else np.argmax(probs_r, axis=1)
                        acc_recenter = (pred_r[valid] == y_true[valid]).mean() if valid.any() else float('nan')
                    except ValueError:
                        pass  # no PC snapshot for this chip -- leave acc_recenter as NaN

                    rows.append({
                        "curve_type": curve_type, "curve_alignment": curve_alignment,
                        "pc_ttp_anchor": pc_ttp_anchor if curve_alignment == "pc_ttp" else "-",
                        "outlier_filter": filter_key_raw, "model": model_key,
                        "held_out_chip": short_name(chip_name),
                        "n_pixels": int(valid.sum()), "acc_baseline": acc_base,
                        "acc_pc_recenter": acc_recenter, "recenter_delta": acc_recenter - acc_base,
                        "shift_norm": shift_norm,
                    })
                    print(f"  [OK] {model_key:38s} filter={filter_key_raw:8s} "
                         f"align={curve_alignment}/{pc_ttp_anchor if curve_alignment=='pc_ttp' else '-':10s} "
                         f"chip={short_name(chip_name):10s} base={acc_base*100:5.1f}% recenter={acc_recenter*100:5.1f}%")

results_df = pd.DataFrame(rows)
print(f"\n{len(results_df)} trained combinations found and evaluated.")

  [*] EXCLUDE_WELL_MAPPING: dropping 0 samples from wells [8] (y_label={8: 'PC'}, y_concentration={8: '0e+00'}) for D20260806_E00_C00_F4500KHz_U_DDM_01_06


/vol/bitbucket/gk225/venv_poc_ddm/lib/python3.12/site-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'loss_scale_optimizer', because it has 74 variables whereas the saved optimizer has 70 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))
/vol/bitbucket/gk225/venv_poc_ddm/lib/python3.12/site-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 70 variables whereas the saved optimizer has 0 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))
I0000 00:00:1786721610.924102 1222222 service.cc:145] XLA service 0x529b2240 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1786721610.924139 1222222 service.cc:153]   StreamExecutor device (0): NVIDIA GeForce GTX 1080, Compute Capability 6.1
I0000 00:00:1786721611.005918 1222222 device_compiler.h:188] Compiled cluster using XLA!  This li

  [*] Saved PC reference embedding -> /vol/bitbucket/gk225/POC_DDM_datasets/POC_DDM_final_nc_subtract/cross_dataset_cv/final_4_chip_clean/pc_reference_embedding_cnn_gru_dual_supcon3_None_ori_curve_wavelet_bior35_norm.joblib
  [OK] cnn_gru_dual_supcon3                   filter=none     align=acquisition_start/-          chip=DDM_01_06  base= 36.1% recenter= 55.7%


/vol/bitbucket/gk225/venv_poc_ddm/lib/python3.12/site-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'loss_scale_optimizer', because it has 82 variables whereas the saved optimizer has 78 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))
/vol/bitbucket/gk225/venv_poc_ddm/lib/python3.12/site-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 78 variables whereas the saved optimizer has 0 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


  [*] Saved PC reference embedding -> /vol/bitbucket/gk225/POC_DDM_datasets/POC_DDM_final_nc_subtract/cross_dataset_cv/final_4_chip_clean/pc_reference_embedding_cnn_gru_dual_supcon3_dann_None_ori_curve_wavelet_bior35_norm.joblib
  [OK] cnn_gru_dual_supcon3_dann              filter=none     align=acquisition_start/-          chip=DDM_01_06  base= 33.2% recenter= 49.2%


...

  [*] Saved PC reference embedding -> /vol/bitbucket/gk225/POC_DDM_datasets/POC_DDM_final_nc_subtract/cross_dataset_cv/final_4_chip_clean/curve_alignment_pc_ttp/anchor_min/pc_reference_embedding_cnn_gru_dual_supcon3_dann_lofo_ae_ori_curve_wavelet_bior35_norm.joblib
  [OK] cnn_gru_dual_supcon3_dann              filter=lofo_ae  align=pc_ttp/min        chip=DDM_01_06  base= 24.2% recenter= 38.9%


/vol/bitbucket/gk225/venv_poc_ddm/lib/python3.12/site-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'loss_scale_optimizer', because it has 86 variables whereas the saved optimizer has 82 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))
/vol/bitbucket/gk225/venv_poc_ddm/lib/python3.12/site-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 82 variables whereas the saved optimizer has 0 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


  [*] Saved PC reference embedding -> /vol/bitbucket/gk225/POC_DDM_datasets/POC_DDM_final_nc_subtract/cross_dataset_cv/final_4_chip_clean/curve_alignment_pc_ttp/anchor_min/pc_reference_embedding_cnn_gru_dual_attn_recon_supcon3_lofo_ae_ori_curve_wavelet_bior35_norm.joblib
  [OK] cnn_gru_dual_attn_recon_supcon3        filter=lofo_ae  align=pc_ttp/min        chip=DDM_01_06  base= 36.5% recenter= 37.9%


/vol/bitbucket/gk225/venv_poc_ddm/lib/python3.12/site-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'loss_scale_optimizer', because it has 94 variables whereas the saved optimizer has 90 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))
/vol/bitbucket/gk225/venv_poc_ddm/lib/python3.12/site-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 90 variables whereas the saved optimizer has 0 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


  [*] Saved PC reference embedding -> /vol/bitbucket/gk225/POC_DDM_datasets/POC_DDM_final_nc_subtract/cross_dataset_cv/final_4_chip_clean/curve_alignment_pc_ttp/anchor_min/pc_reference_embedding_cnn_gru_dual_attn_recon_supcon3_dann_lofo_ae_ori_curve_wavelet_bior35_norm.joblib
  [OK] cnn_gru_dual_attn_recon_supcon3_dann   filter=lofo_ae  align=pc_ttp/min        chip=DDM_01_06  base= 21.8% recenter= 18.9%
  [*] EXCLUDE_WELL_MAPPING: dropping 0 samples from wells [8] (y_label={8: 'PC'}, y_concentration={8: '0e+00'}) for D20260806_E00_C00_F4500KHz_U_DDM_01_06
[!] No saved resampler at /vol/bitbucket/gk225/POC_DDM_datasets/POC_DDM_final_nc_subtract/cross_dataset_cv/final_4_chip_clean/curve_alignment_pc_ttp/anchor_percentile/cross_dataset_resampler_classification_performances_ori_curve_wavelet_bior35_norm.joblib. Run 04_cross_dataset_training.py --train_full for this group/curve_type first.

16 trained combinations found and evaluated.


## 4. Results

Per-(chip, combination) rows first, then aggregated by combination (mean across the
4 chips) -- sorted so the biggest recentering wins float to the top. A combination
only appears here once it's actually been trained; re-run the sweep above after the
job finishes to pick up more.

In [5]:
results_df.sort_values(["model", "outlier_filter", "curve_alignment", "held_out_chip"])

,curve_type,curve_alignment,pc_ttp_anchor,outlier_filter,model,held_out_chip,n_pixels,acc_baseline,acc_pc_recenter,recenter_delta,shift_norm
6,ori_curve_wavelet_bior35_norm,acquisition_start,-,lofo_ae,cnn_gru_dual_attn_recon_supcon3,DDM_01_06,14507,0.284759,0.299442,0.014683,27.940077
14,ori_curve_wavelet_bior35_norm,pc_ttp,min,lofo_ae,cnn_gru_dual_attn_recon_supcon3,DDM_01_06,14507,0.364514,0.378507,0.013993,7.401068
2,ori_curve_wavelet_bior35_norm,acquisition_start,-,none,cnn_gru_dual_attn_recon_supcon3,DDM_01_06,14507,0.355346,0.514579,0.159233,25.846266
10,ori_curve_wavelet_bior35_norm,pc_ttp,min,none,cnn_gru_dual_attn_recon_supcon3,DDM_01_06,14507,0.386434,0.334390,-0.052044,15.913676
7,ori_curve_wavelet_bior35_norm,acquisition_start,-,lofo_ae,cnn_gru_dual_attn_recon_supcon3_dann,DDM_01_06,14507,0.233198,0.344799,0.111601,16.583269
15,ori_curve_wavelet_bior35_norm,pc_ttp,min,lofo_ae,cnn_gru_dual_attn_recon_supcon3_dann,DDM_01_06,14507,0.217550,0.188599,-0.028952,11.010244
3,ori_curve_wavelet_bior35_norm,acquisition_start,-,none,cnn_gru_dual_attn_recon_supcon3_dann,DDM_01_06,14507,0.210312,0.201420,-0.008892,10.079744
11,ori_curve_wavelet_bior35_norm,pc_ttp,min,none,cnn_gru_dual_attn_recon_supcon3_dann,DDM_01_06,14507,0.287999,0.293238,0.005239,16.249704
4,ori_curve_wavelet_bior35_norm,acquisition_start,-,lofo_ae,cnn_gru_dual_supcon3,DDM_01_06,14507,0.325498,0.500586,0.175088,2.334917
12,ori_curve_wavelet_bior35_norm,pc_ttp,min,lofo_ae,cnn_gru_dual_supcon3,DDM_01_06,14507,0.272972,0.335493,0.062522,4.731965


In [6]:
summary = (results_df
    .groupby(["model", "outlier_filter", "curve_alignment", "pc_ttp_anchor"])
    .agg(n_chips=("held_out_chip", "nunique"),
        acc_baseline=("acc_baseline", "mean"),
        acc_pc_recenter=("acc_pc_recenter", "mean"),
        recenter_delta=("recenter_delta", "mean"))
    .reset_index()
    .sort_values("acc_baseline", ascending=False))
summary

,model,outlier_filter,curve_alignment,pc_ttp_anchor,n_chips,acc_baseline,acc_pc_recenter,recenter_delta
3,cnn_gru_dual_attn_recon_supcon3,none,pc_ttp,min,1,0.386434,0.334390,-0.052044
12,cnn_gru_dual_supcon3_dann,lofo_ae,acquisition_start,-,1,0.385538,0.515337,0.129799
1,cnn_gru_dual_attn_recon_supcon3,lofo_ae,pc_ttp,min,1,0.364514,0.378507,0.013993
10,cnn_gru_dual_supcon3,none,acquisition_start,-,1,0.360998,0.557455,0.196457
2,cnn_gru_dual_attn_recon_supcon3,none,acquisition_start,-,1,0.355346,0.514579,0.159233
14,cnn_gru_dual_supcon3_dann,none,acquisition_start,-,1,0.332391,0.491625,0.159233
8,cnn_gru_dual_supcon3,lofo_ae,acquisition_start,-,1,0.325498,0.500586,0.175088
15,cnn_gru_dual_supcon3_dann,none,pc_ttp,min,1,0.303440,0.405322,0.101882
7,cnn_gru_dual_attn_recon_supcon3_dann,none,pc_ttp,min,1,0.287999,0.293238,0.005239
0,cnn_gru_dual_attn_recon_supcon3,lofo_ae,acquisition_start,-,1,0.284759,0.299442,0.014683


In [7]:
summary.sort_values("recenter_delta", ascending=False)

,model,outlier_filter,curve_alignment,pc_ttp_anchor,n_chips,acc_baseline,acc_pc_recenter,recenter_delta
10,cnn_gru_dual_supcon3,none,acquisition_start,-,1,0.360998,0.557455,0.196457
8,cnn_gru_dual_supcon3,lofo_ae,acquisition_start,-,1,0.325498,0.500586,0.175088
2,cnn_gru_dual_attn_recon_supcon3,none,acquisition_start,-,1,0.355346,0.514579,0.159233
14,cnn_gru_dual_supcon3_dann,none,acquisition_start,-,1,0.332391,0.491625,0.159233
13,cnn_gru_dual_supcon3_dann,lofo_ae,pc_ttp,min,1,0.241539,0.389398,0.147860
11,cnn_gru_dual_supcon3,none,pc_ttp,min,1,0.277177,0.412835,0.135659
12,cnn_gru_dual_supcon3_dann,lofo_ae,acquisition_start,-,1,0.385538,0.515337,0.129799
4,cnn_gru_dual_attn_recon_supcon3_dann,lofo_ae,acquisition_start,-,1,0.233198,0.344799,0.111601
15,cnn_gru_dual_supcon3_dann,none,pc_ttp,min,1,0.303440,0.405322,0.101882
9,cnn_gru_dual_supcon3,lofo_ae,pc_ttp,min,1,0.272972,0.335493,0.062522
